# Notebook 3: Hypothesis Space & Inductive Bias

**Difficulty:** ⭐⭐⭐ | **Estimated Time:** 2 hours  
**Week 5 — ML Fundamentals & Linear Regression**

---

## What You Will Learn

| Concept | One-line Summary |
|---|---|
| Hypothesis Space | The complete set of all models your algorithm can possibly produce |
| Inductive Bias | The built-in assumptions that make learning possible |
| No Free Lunch Theorem | No model wins on every problem — biases matter |
| Occam's Razor | When two models fit equally well, prefer the simpler one |

## 1. Why This Matters

Every time you pick a model — linear regression, a decision tree, KNN — you are making a **bet**.  
You are betting that the pattern hiding in your data looks like something your model's family of functions can actually express.

If you bet wrong, no amount of training data or clever optimisation will save you.  
A straight-line model will never fit a circle, no matter how long you train it.

Understanding hypothesis space and inductive bias gives you a principled way to make that bet — one informed by domain knowledge rather than guesswork.

## 2. Real-World Analogy — The Real-Estate Agent's Rulebook

Imagine you are a new real-estate agent trying to price houses. Before you see any data, you have a **rulebook** of pricing strategies you are willing to consider:

- **Rulebook A (Linear model):** "Price = base price + some fixed amount per square foot." Simple, fast, but can only express straight-line relationships.
- **Rulebook B (Polynomial model):** "Price = base + a·sqft + b·sqft² + ..." Can express curves — a small house might be underpriced AND a very large house might also be underpriced relative to mid-sized ones.
- **Rulebook C (Neural network):** "Price = whatever complex function fits the data." Practically any pricing rule is on the table.

The rulebook you carry is your **hypothesis space** — it is the universe of answers you are even willing to consider.  
The fact that you *prefer* certain rules over others before seeing any data — e.g., "I'll assume price grows linearly with size unless proven otherwise" — is your **inductive bias**.

> **Key insight:** A larger rulebook (bigger hypothesis space) is not automatically better. It means more ways to be right, but also more ways to be spectacularly wrong.

## 3. Hypothesis Space — Plain English

The **hypothesis space** H is the set of all functions your model *can* represent, given its architecture:

| Model | Hypothesis Space |
|---|---|
| Linear Regression | All lines (2D), all hyperplanes (higher D) |
| Degree-2 Polynomial | All parabolas |
| Degree-d Polynomial | All polynomials up to degree d |
| Decision Tree (depth k) | All axis-aligned partitions with at most 2^k leaves |
| KNN | Locally piecewise-constant functions |
| Deep Neural Network | Theoretically any continuous function (Universal Approximation Theorem) |

Learning = **searching this space** for the function h ∈ H that best fits the training data.

$$h^* = \arg\min_{h \in H} \sum_{i=1}^{n} L(h(x_i), y_i)$$

where L is your loss function (e.g., squared error).

In [ ]:
# ── Cell 1: Imports ──────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

# Reproducibility: fix the random seed so every run gives identical plots
np.random.seed(42)

# Global style — makes all plots look consistent
plt.rcParams.update({
    'figure.dpi': 110,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11
})
print("Libraries loaded.")

## 4. Visualising the Hypothesis Space — Many Functions Before Seeing Data

Before any training data arrives, a linear model considers **every line** equally plausible.  
That is its full hypothesis space. The plot below draws 60 random lines — each one is a valid hypothesis.

In [ ]:
# ── Cell 2: Hypothesis Space Visualisation ───────────────────────────────────
# True underlying relationship: price grows roughly linearly with house size
n = 40
size_true = np.linspace(500, 3000, n)          # House sizes in sq ft
price_true = 50 + 0.15 * size_true              # True line (unknown to model)
noise = np.random.normal(0, 15, n)              # Realistic measurement noise
price_obs = price_true + noise                  # Observed (noisy) prices in $k

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Left: the full hypothesis space (before data) ────────────────────────────
ax = axes[0]
x_plot = np.linspace(500, 3000, 200)
for _ in range(60):
    # Draw random slope and intercept — each pair is one hypothesis
    slope = np.random.uniform(-0.05, 0.35)
    intercept = np.random.uniform(-50, 150)
    ax.plot(x_plot, intercept + slope * x_plot,
            color='steelblue', alpha=0.15, linewidth=0.8)
ax.set(title='Hypothesis Space (before data)\n60 random lines — all equally possible',
       xlabel='House Size (sq ft)', ylabel='Price ($k)')

# ── Right: data narrows the space to one winner ───────────────────────────────
ax = axes[1]
# Show a few remaining 'plausible' hypotheses after seeing some data
for _ in range(15):
    slope = np.random.normal(0.15, 0.02)        # Concentrated near true slope
    intercept = np.random.normal(50, 8)         # Concentrated near true intercept
    ax.plot(x_plot, intercept + slope * x_plot,
            color='steelblue', alpha=0.3, linewidth=0.9)
ax.scatter(size_true, price_obs, color='tomato', s=30, zorder=5, label='Data')
ax.plot(x_plot, 50 + 0.15 * x_plot, color='black', linewidth=2,
        linestyle='--', label='Best fit (winner)')
ax.set(title='After Seeing Data\nSpace collapses to best-fitting hypothesis',
       xlabel='House Size (sq ft)', ylabel='Price ($k)')
ax.legend()

plt.suptitle('Hypothesis Space: All Bets → One Winner', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 5. Polynomial Degree = Expanding the Hypothesis Space

Increasing polynomial degree is one of the clearest ways to see hypothesis spaces of different sizes.  
Watch what happens as we go from degree 1 (just a line) up to degree 10 (wild curves).

In [ ]:
# ── Cell 3: Polynomial Models of Increasing Degree ───────────────────────────
# True relationship has a gentle curve: price levels off at very large houses
n = 30
size = np.sort(np.random.uniform(500, 3000, n))
# True function: mild quadratic — grows fast then levels off
price = 20 + 0.25 * size - 0.00003 * size**2 + np.random.normal(0, 12, n)

x_plot = np.linspace(500, 3000, 300).reshape(-1, 1)
degrees = [1, 2, 5, 10]
colors  = ['#2196F3', '#4CAF50', '#FF9800', '#F44336']

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
axes = axes.flatten()

for ax, deg, col in zip(axes, degrees, colors):
    # Pipeline: polynomial feature expansion → linear regression
    model = Pipeline([
        ('poly', PolynomialFeatures(degree=deg, include_bias=False)),
        ('lr',   LinearRegression())
    ])
    model.fit(size.reshape(-1, 1), price)
    y_pred = model.predict(x_plot)

    # Training MSE measures how well this degree fits the 30 points
    train_mse = mean_squared_error(price, model.predict(size.reshape(-1, 1)))

    ax.scatter(size, price, color='gray', s=25, zorder=5, label='Data')
    ax.plot(x_plot, y_pred, color=col, linewidth=2.5,
            label=f'Degree {deg} | Train MSE={train_mse:.1f}')
    ax.set(title=f'Degree-{deg} Polynomial',
           xlabel='Size (sq ft)', ylabel='Price ($k)',
           ylim=(price.min()-20, price.max()+20))
    ax.legend(fontsize=9)

plt.suptitle('Hypothesis Space Grows with Polynomial Degree\n'
             'Degree 1: only lines  |  Degree 10: nearly any curve',
             fontsize=12)
plt.tight_layout()
plt.show()

print("\nNote: Degree 10 has near-zero training error but wild oscillations —")
print("it is memorising noise, not learning the pattern.")

## 6. Inductive Bias — Plain English

The hypothesis space tells us *what* a model can learn. Inductive bias tells us *which* hypotheses the model prefers when multiple ones fit the data equally well.

| Model | Inductive Bias (built-in assumption) |
|---|---|
| Linear Regression | The target is a linear function of the inputs |
| Decision Tree | The input space can be split by axis-aligned thresholds |
| KNN (k=5) | Nearby points in feature space have similar outputs |
| Naive Bayes | Features are conditionally independent given the class |
| Ridge Regression | Prefer smaller coefficients (L2 penalty) |

**Why do we need inductive bias?**  
Without it, any function that perfectly fits the training data would be equally valid — including a function that memorises all 30 training examples and outputs garbage everywhere else.  
Inductive bias is what makes **generalisation** possible.

In [ ]:
# ── Cell 4: Inductive Bias Demo — Three Models, Same Data ────────────────────
# Sinusoidal data: none of these models is perfectly suited to it
# We will see each model's bias shining through in how it fails
n = 60
x_sin = np.sort(np.random.uniform(0, 2 * np.pi, n))
# True pattern: a sine wave (imagine seasonal price swings by month)
y_sin = np.sin(x_sin) + np.random.normal(0, 0.15, n)

x_plot_sin = np.linspace(0, 2 * np.pi, 300).reshape(-1, 1)

# Three models with very different inductive biases
lin_model  = LinearRegression()
knn_model  = KNeighborsRegressor(n_neighbors=3)  # Local smoothness bias
tree_model = DecisionTreeRegressor(max_depth=4)   # Axis-aligned split bias

models      = [lin_model,   knn_model,    tree_model]
model_names = ['Linear Regression\n(bias: linear relationship)',
               'KNN (k=3)\n(bias: local smoothness)',
               'Decision Tree (depth 4)\n(bias: rectangular regions)']
model_cols  = ['#2196F3', '#4CAF50', '#FF9800']

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

for ax, model, name, col in zip(axes, models, model_names, model_cols):
    model.fit(x_sin.reshape(-1, 1), y_sin)
    y_pred = model.predict(x_plot_sin)
    mse    = mean_squared_error(y_sin, model.predict(x_sin.reshape(-1, 1)))

    ax.scatter(x_sin, y_sin, color='gray', s=18, alpha=0.7, label='Data')
    ax.plot(x_plot_sin, y_pred, color=col, linewidth=2.5,
            label=f'Train MSE={mse:.3f}')
    # True sine for reference
    ax.plot(x_plot_sin, np.sin(x_plot_sin), color='black',
            linewidth=1.5, linestyle='--', alpha=0.5, label='True sin(x)')
    ax.set(title=name, xlabel='x', )
    ax.legend(fontsize=8)

axes[0].set_ylabel('y')
plt.suptitle('Same Data, Different Inductive Biases — Each Model "Sees" the Pattern Differently',
             fontsize=11)
plt.tight_layout()
plt.show()

## 7. The No Free Lunch Theorem

> **Theorem (Wolpert, 1996):** Averaged over all possible problems, every learning algorithm performs equally well (or equally poorly).

### What this means in practice

There is no universally best model. A model that wins on linear data will lose on sinusoidal data.  
A model that wins on sinusoidal data will lose on step-function data.

**Implication for house price prediction:**  
If you know that price grows roughly linearly with size (domain knowledge), you should lean towards linear models.  
If you suspect complex interactions (neighbourhood, school district, size all interact), you need a more flexible model.

**Domain knowledge is the cheat code.** It lets you pick a model whose bias aligns with the true structure of your problem — which is exactly what 'good' inductive bias means.

In [ ]:
# ── Cell 5: No Free Lunch Demo ────────────────────────────────────────────────
# Three very different data patterns — each model wins on its home turf
np.random.seed(42)
n = 80
x = np.sort(np.random.uniform(0, 10, n))

# Pattern 1: perfectly linear relationship
y_linear = 2 * x + 5 + np.random.normal(0, 1, n)

# Pattern 2: sinusoidal (seasonal effects, neighbourhood cycles)
y_sine = np.sin(x) * 10 + np.random.normal(0, 0.8, n)

# Pattern 3: step function (discrete zone-based pricing in a city)
y_step = (np.digitize(x, bins=[2, 5, 7]) * 10) + np.random.normal(0, 0.8, n)

patterns      = [('Linear Data',      y_linear),
                 ('Sinusoidal Data',   y_sine),
                 ('Step-Function Data', y_step)]
model_list    = [
    ('Linear',   LinearRegression()),
    ('KNN k=5',  KNeighborsRegressor(n_neighbors=5)),
    ('Tree d=4', DecisionTreeRegressor(max_depth=4))
]
cols = ['#2196F3', '#4CAF50', '#FF9800']

fig, axes = plt.subplots(3, 3, figsize=(14, 11))
x_plot_nfl = np.linspace(0, 10, 300).reshape(-1, 1)

for row_idx, (pat_name, y_pat) in enumerate(patterns):
    for col_idx, ((mod_name, model), col) in enumerate(zip(model_list, cols)):
        ax = axes[row_idx][col_idx]
        model.fit(x.reshape(-1, 1), y_pat)
        mse = mean_squared_error(y_pat, model.predict(x.reshape(-1, 1)))
        y_fit = model.predict(x_plot_nfl)

        ax.scatter(x, y_pat, color='gray', s=12, alpha=0.6)
        ax.plot(x_plot_nfl, y_fit, color=col, linewidth=2)
        # Highlight the winner (lowest MSE per row) with a gold border
        ax.set(title=f'{mod_name}\nMSE={mse:.2f}')
        if col_idx == 0:
            ax.set_ylabel(pat_name, fontsize=10, fontweight='bold')

plt.suptitle('No Free Lunch: Each Model Wins on Its Home Turf, Fails on Others',
             fontsize=12)
plt.tight_layout()
plt.show()

# Summarise MSEs in a clean table
rows = []
for pat_name, y_pat in patterns:
    row = {'Data Pattern': pat_name}
    for mod_name, model in model_list:
        model.fit(x.reshape(-1, 1), y_pat)
        row[mod_name] = round(mean_squared_error(y_pat,
                              model.predict(x.reshape(-1, 1))), 2)
    rows.append(row)

df_nfl = pd.DataFrame(rows).set_index('Data Pattern')
print("\nTraining MSE Summary (lower = better for that row):")
print(df_nfl.to_string())

## 8. Occam's Razor in Machine Learning

> **Occam's Razor:** Among all hypotheses that explain the observations equally well, prefer the simplest.

In ML terms: if a degree-2 polynomial and a degree-8 polynomial both achieve similar training error, prefer the degree-2.  
Why? Because the degree-8 polynomial has many more ways to overfit. Its extra complexity is not buying you anything.

**Regularisation is Occam's Razor implemented mathematically.**  
Ridge regression adds a penalty term `λ‖w‖²` to the loss function, explicitly penalising large, complex coefficients and forcing the model toward simpler solutions.

In [ ]:
# ── Cell 6: Occam's Razor — Same Training MSE, Very Different Complexity ──────
np.random.seed(7)
n_train = 25
# Ground truth: a gently curved relationship (price vs size)
x_train = np.sort(np.random.uniform(500, 3000, n_train))
y_train = 30 + 0.12 * x_train - 0.000015 * x_train**2 + np.random.normal(0, 10, n_train)

# Validation set: 200 points drawn from the same distribution
x_val = np.sort(np.random.uniform(500, 3000, 200))
y_val = 30 + 0.12 * x_val - 0.000015 * x_val**2 + np.random.normal(0, 10, 200)

x_plot_occ = np.linspace(500, 3000, 300).reshape(-1, 1)

# Model A: degree 2 — simple
model_d2 = Pipeline([('poly', PolynomialFeatures(2)), ('lr', LinearRegression())])
# Model B: degree 8 — complex
model_d8 = Pipeline([('poly', PolynomialFeatures(8)), ('lr', LinearRegression())])

model_d2.fit(x_train.reshape(-1, 1), y_train)
model_d8.fit(x_train.reshape(-1, 1), y_train)

# Compare training and validation MSEs
mse_d2_train = mean_squared_error(y_train, model_d2.predict(x_train.reshape(-1,1)))
mse_d2_val   = mean_squared_error(y_val,   model_d2.predict(x_val.reshape(-1,1)))
mse_d8_train = mean_squared_error(y_train, model_d8.predict(x_train.reshape(-1,1)))
mse_d8_val   = mean_squared_error(y_val,   model_d8.predict(x_val.reshape(-1,1)))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, model, deg, mse_tr, mse_v, col in [
    (axes[0], model_d2, 2, mse_d2_train, mse_d2_val, '#4CAF50'),
    (axes[1], model_d8, 8, mse_d8_train, mse_d8_val, '#F44336')
]:
    y_fit = model.predict(x_plot_occ)
    ax.scatter(x_train, y_train, color='black', s=30, zorder=5, label='Training data')
    ax.plot(x_plot_occ, y_fit, color=col, linewidth=2.5)
    ax.set(title=f'Degree-{deg} Polynomial\n'
                 f'Train MSE={mse_tr:.1f}  |  Val MSE={mse_v:.1f}',
           xlabel='Size (sq ft)', ylabel='Price ($k)',
           ylim=(y_train.min()-30, y_train.max()+30))
    ax.legend(fontsize=9)

plt.suptitle("Occam's Razor: Similar Training Error, But Degree-2 Generalises Far Better",
             fontsize=11)
plt.tight_layout()
plt.show()

print(f"Degree-2  → Train MSE: {mse_d2_train:.1f}  | Validation MSE: {mse_d2_val:.1f}")
print(f"Degree-8  → Train MSE: {mse_d8_train:.1f}  | Validation MSE: {mse_d8_val:.1f}")
print("\nOccam's winner: Degree-2 (simpler, similar training fit, much better on new data)")

## 9. Regularisation as Occam's Razor

Ridge regression directly implements Occam's Razor by penalising model complexity:

$$\text{Loss}_{\text{Ridge}} = \underbrace{\sum_{i=1}^n (y_i - \hat{y}_i)^2}_{\text{fit the data}} + \underbrace{\lambda \sum_{j=1}^p w_j^2}_{\text{stay simple}}$$

- Small λ → prefer fitting data (allows complexity)
- Large λ → prefer simplicity (shrinks coefficients toward zero)

This is the algorithmic equivalent of "don't make it more complicated than it needs to be."

In [ ]:
# ── Cell 7: Regularisation Tames a Complex Model ──────────────────────────────
lambdas = [0.0001, 1, 100, 10000]   # Increasing regularisation strength
colors_reg = ['#E53935', '#FB8C00', '#43A047', '#1E88E5']

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
axes = axes.flatten()

for ax, lam, col in zip(axes, lambdas, colors_reg):
    # Degree-8 polynomial with Ridge regularisation
    model_ridge = Pipeline([
        ('poly', PolynomialFeatures(8)),
        ('ridge', Ridge(alpha=lam))   # alpha in sklearn = λ in our formula
    ])
    model_ridge.fit(x_train.reshape(-1, 1), y_train)
    y_fit  = model_ridge.predict(x_plot_occ)
    mse_tr = mean_squared_error(y_train,
                 model_ridge.predict(x_train.reshape(-1, 1)))
    mse_v  = mean_squared_error(y_val,
                 model_ridge.predict(x_val.reshape(-1, 1)))

    ax.scatter(x_train, y_train, color='black', s=25, zorder=5)
    ax.plot(x_plot_occ, y_fit, color=col, linewidth=2.5)
    ax.set(title=f'λ = {lam}\nTrain MSE={mse_tr:.1f}  Val MSE={mse_v:.1f}',
           xlabel='Size (sq ft)', ylabel='Price ($k)',
           ylim=(y_train.min()-30, y_train.max()+30))

plt.suptitle('Ridge Regularisation on Degree-8 Polynomial\n'
             'Higher λ → Simpler (flatter) curve → Occam in action', fontsize=11)
plt.tight_layout()
plt.show()

## 10. House Price — Putting It All Together

Let us walk through the entire narrative with realistic house price features.

- **Hypothesis:** "Price depends on size and bedrooms."
- **Linear model assumption (inductive bias):** price changes at a constant rate per sq ft.
- **More complex hypothesis:** "price per sq ft decreases once a house has more than 4 bedrooms" — this requires a non-linear model or interaction features.

In [ ]:
# ── Cell 8: House Price — Multi-Feature Hypothesis Space ──────────────────────
np.random.seed(0)
N = 200

# Simulate a small house-price dataset
sqft      = np.random.uniform(600, 3500, N)
bedrooms  = np.random.randint(1, 7, N)           # 1–6 bedrooms

# True pricing rule: non-linear — a house with >4 bedrooms gets a DISCOUNT
# (too many bedrooms for a given sq ft = choppy layout)
bonus     = np.where(bedrooms <= 4, bedrooms * 8, bedrooms * 2)
price_2d  = 40 + 0.13 * sqft + bonus + np.random.normal(0, 12, N)

df = pd.DataFrame({'sqft': sqft, 'bedrooms': bedrooms, 'price': price_2d})

# ── Model A: linear in sqft only ──────────────────────────────────────────────
X_simple = df[['sqft']]
lr_simple = LinearRegression().fit(X_simple, df['price'])

# ── Model B: linear in sqft + bedrooms ────────────────────────────────────────
X_full = df[['sqft', 'bedrooms']]
lr_full = LinearRegression().fit(X_full, df['price'])

# ── Model C: interactions + polynomial (captures the bedroom penalty) ──────────
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X_full)
lr_poly = LinearRegression().fit(X_poly, df['price'])

mse_simple = mean_squared_error(df['price'], lr_simple.predict(X_simple))
mse_full   = mean_squared_error(df['price'], lr_full.predict(X_full))
mse_poly   = mean_squared_error(df['price'], lr_poly.predict(X_poly))

print("Model Comparison — House Price Prediction")
print("-" * 50)
print(f"Linear (sqft only):              MSE = {mse_simple:.1f}")
print(f"Linear (sqft + bedrooms):        MSE = {mse_full:.1f}")
print(f"Degree-2 poly (all interactions): MSE = {mse_poly:.1f}")
print("\nThe polynomial model captures the bedroom penalty (non-linear effect)")
print("— its larger hypothesis space contains the true function.")

In [ ]:
# ── Cell 9: Visualise the Bedroom Penalty Effect ──────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: price vs sqft, coloured by number of bedrooms
ax = axes[0]
sc = ax.scatter(df['sqft'], df['price'], c=df['bedrooms'],
                cmap='RdYlGn_r', s=25, alpha=0.7)
plt.colorbar(sc, ax=ax, label='Bedrooms')
ax.set(title='House Prices — Colour = Number of Bedrooms\n'
             '(High bedroom count ≠ higher price)',
       xlabel='Sq Ft', ylabel='Price ($k)')

# Right: average price by bedroom count — shows the non-linearity
ax = axes[1]
avg_by_bed = df.groupby('bedrooms')['price'].mean()
ax.bar(avg_by_bed.index, avg_by_bed.values,
       color=['#4CAF50' if b <= 4 else '#F44336' for b in avg_by_bed.index],
       edgecolor='white')
ax.set(title='Average Price by Bedroom Count\nGreen=linear regime  Red=discount regime',
       xlabel='Number of Bedrooms', ylabel='Average Price ($k)')
ax.axvline(4.5, color='black', linestyle='--', linewidth=1.5)
ax.text(4.6, avg_by_bed.max() * 0.95, 'Discount\nkicks in', fontsize=9)

plt.suptitle('Domain Knowledge: The True Rule Has a Non-Linear Bedroom Effect\n'
             'A linear hypothesis space cannot represent this', fontsize=11)
plt.tight_layout()
plt.show()

## 11. Summary — Key Takeaways

| Concept | One-line Rule of Thumb |
|---|---|
| Hypothesis Space | Your model can only learn what its space contains — choose wisely |
| Inductive Bias | Bias is a feature, not a bug — it makes generalisation possible |
| No Free Lunch | No universal champion — let domain knowledge guide model choice |
| Occam's Razor | Complexity must earn its place — validate on held-out data |
| Regularisation | The principled, mathematical implementation of Occam's Razor |

### The Decision Loop

```
Understand the problem domain
       ↓
Choose a hypothesis space that can express the true pattern
       ↓
Pick a model whose inductive bias aligns with domain knowledge
       ↓
Validate — prefer simpler models when validation error is similar
```

## 12. Self-Check Questions

Test your understanding. Try to answer each question before expanding the answer.

---

**Question 1:** You choose a degree-10 polynomial for a dataset that has 12 training points. What could go wrong? Why?

<details>
<summary>Click to reveal answer</summary>

A degree-10 polynomial has 11 coefficients. With only 12 training points, the model has almost as many parameters as data points. This means it can (and likely will) pass through every training point exactly — achieving near-zero training error — but the curve will oscillate wildly between points. On any new data, the predictions will be terrible. This is classic **overfitting**: the model memorises training noise instead of learning the underlying pattern. The hypothesis space (degree-10 polynomials) is far larger than what is needed, and there is not enough data to constrain it to the right region.

</details>

---

**Question 2:** KNN assumes "nearby points have similar outputs." On what kind of data would this assumption break down?

<details>
<summary>Click to reveal answer</summary>

KNN's inductive bias (local smoothness) breaks down whenever nearby points can have very different outputs:

1. **High-frequency oscillations:** In a sine wave with many cycles over a short range, two adjacent x-values can produce very different y-values. KNN will average them and miss the pattern.
2. **Sharp discontinuities / step functions:** A house price that jumps dramatically at a school-district boundary — two houses 50m apart can have very different prices. KNN averages them and smooths out the jump.
3. **High-dimensional sparse data:** In many dimensions, "nearby" becomes meaningless (curse of dimensionality) — all points are approximately equally far apart.
4. **Class-imbalanced boundary regions:** When the boundary between two regions is very thin, nearest neighbours may cross the boundary.

The key insight: if the local smoothness assumption does not hold for your data, KNN's bias actively hurts you.

</details>

---

**Question 3:** A linear regression model fits your house price data with MSE=5. A degree-8 polynomial fits with MSE=4.9. Which would you prefer and why?

<details>
<summary>Click to reveal answer</summary>

**Prefer the linear regression model**, by Occam's Razor. Here is the reasoning:

- The improvement from 5.0 to 4.9 is tiny (2% reduction in MSE). 
- The degree-8 polynomial has 9 coefficients vs 2 for linear regression — it is dramatically more complex.
- The MSEs quoted are **training MSEs**. The degree-8 polynomial has much more capacity to memorise noise, so its validation MSE is almost certainly higher than the linear model's.
- The linear model's inductive bias (linearity) is often a safe assumption for house prices as a first-pass model.

**The right question is always: which model has lower *validation* MSE?** If you have not checked validation performance, do that first. But given only training MSE, Occam's Razor strongly favours the simpler model.

</details>